In [8]:
import tqdm 
import sys
import pandas as pd
sys.path.append('../kaggle_prediction_library/') 
from web_scraping.torvik_player_scraping_functions import get_html_from_torvik_players, get_data_from_html

### Scrape the Data

In [9]:
all_first_round_dates = [
    "20080321", "20080320", 
    "20090320", "20090319", 
    "20100319", "20100318",
    "20110318", "20110317", 
    "20120316", "20120315", 
    "20130322", "20130321",
    "20140321", "20140320", 
    "20150320", "20150319", 
    "20160318", "20160317",
    "20170317", "20170316", 
    "20180316", "20180315", 
    "20190322", "20190321",
    "20200320", "20200319",    
    "20210319", "20210320", 
    "20220318", "20220317",
    "20230317", "20230316", 
    "20240322", "20240321"
]

# note this is the day before the first four, for safety
day_before_tournament_start_dict = {
    '2008': '20080317', '2009': '20090316', '2010': '20100315',
    '2011': '20110314', '2012': '20120312', '2013': '20130317',
    '2014': '20140317', '2015': '20150316', '2016': '20160314',
    '2017': '20170313', '2018': '20180312', '2019': '20190318',
    '2020': '20200316', '2021': '20210316', '2022': '20220314',
    '2023': '20230313', '2024': '20240318'
}

day_before_tournament_start_df = pd.DataFrame(list(day_before_tournament_start_dict.items()), columns=['year', 'day_before_tourney_start'])
day_before_tournament_start_df['season_start_date'] = (day_before_tournament_start_df['year'].astype(int) - 1).astype(str) + '1101'


In [10]:
all_dfs = []

for index, row in tqdm.tqdm(day_before_tournament_start_df.head(1).iterrows()):
    
    start = row["season_start_date"]
    end = row["day_before_tourney_start"]
    year = row["year"]

    html = get_html_from_torvik_players(year, start, end, 1)
    tmp_df = get_data_from_html(html)
    tmp_df["Season"] = year

    all_dfs.append(tmp_df)


0it [00:00, ?it/s]

Loading URL: https://barttorvik.com/playerstat.php?link=y&sIndex=53&minmin=5&year=2008&start=20071101&end=20080317
Clicked on 'Games' column successfully.
'Show 100 more' clicked (1/1)


1it [00:37, 37.26s/it]


In [11]:
final_df = pd.concat(all_dfs, axis=0)
final_df = final_df[final_df["Min%"].notnull()]


### Map to the Kaggle Ids 

In [27]:
final_df = pd.read_csv("../data/sky_data/torvik_player_data_2008_2024.csv")

In [28]:
final_kaggle_torvik_mapping = pd.read_csv("../data/sky_data/mappings/kaggle_torvik_mapping.csv")

In [26]:
final_kaggle_torvik_mapping

,Unnamed: 0,Kaggle_Team,Final_Torvik_Team
0,0,UNC Asheville,UNC Asheville
1,1,Arizona,Arizona
2,2,Arizona St,Arizona St.
3,3,C Michigan,Central Michigan
4,4,California,California
...,...,...,...
262,262,Kennesaw,Kennesaw St.
263,263,McNeese St,McNeese St.
264,264,Samford,Samford
265,265,Stetson,Stetson


In [37]:
teams = pd.read_csv("../data/MTeams.csv")

In [ ]:
final_df["Final_Torvik_Team"] = final_df["Team"]
final_df = final_df.merge(final_kaggle_torvik_mapping, how="inner", on=["Final_Torvik_Team"])

In [39]:
final_df["TeamName"] = final_df["Kaggle_Team"]

In [40]:
final_df = final_df.merge(teams[["TeamID", "TeamName"]], how="left", on=["TeamName"])

In [42]:
final_df.to_csv("../data/sky_data/torvik_player_data_2008_2024.csv")